<a href="https://colab.research.google.com/github/donkey-king-kong/CSCI-4907---NLU/blob/main/(GWU)_CSCI_4907_NLU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecting Hate Speech in Tweets - Cyberbullying
Group Members:
> - Zhan You Lau
> - Yu Chen Law
> - Kieran E Kai Voo
> - Joshua, Tse Ern Foo

# Introduction
Social media has become ubiquitous in our everyday communication which contributes to the increased prevalence of cyberbullying. However, cyberbullying detection is particularly challenging in a multilabel setting as harmful content may belong to multiple overlapping categories like threats, insults, or implicit aggression. These categories often rely on subtle linguistic cues such as sarcasm and contextual ambiguity. This makes accurate classification difficult.

## Problem Formulation
Our project examines how different modeling approaches handle the complexities of multilabel cyberbullying detection as well as evaluates their strengths and limitations beyond overall performance metrics.

### Key Objective
- Perform data preparation, preprocessing, and exploratory analysis
- Implement classical machine learning models and Bi-LSTM for comparison
- Evaluate and compare model performance using multilabel metrics
- Conduct structured error analysis to examine linguistic failure cases

## Trigger Warning
These tweets either describe a bullying event or are the offense themselves, therefore explore it to the point where you feel comfortable.

In [1]:
import pandas as pd
df = pd.read_csv('cyberbullying_tweets.csv')
df.head()

,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


## Data Preparation & Cleaning
The following actions are performed to the data present in our dataset to bring a uniformity and remove any stray characters to improve effeciency of our model.

In [2]:
import string
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [11]:
# Function to make text lowercase
def make_lowercase(text):
    lower = text.lower()
    return lower

# Function to remove mentions (@username) from text
def remove_mentions(text):
    clean_text = re.sub(r'@\S+', '', text)
    return clean_text

# Function to remove punctuation from text
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    clean_text = text.translate(translator)
    return clean_text

# Function to remove URLs from text
def remove_url(text):
    clean_text = re.sub(r'https?://\S+|www\.\S+', '', text)
    return clean_text

# Function to remove extra whitespaces from text
def remove_extraspace(text):
    words = text.split()
    updated_text = " ".join(words)
    return updated_text

# Function to remove stopwords from text
def remove_stopwords(text):
    tweet_words = text.split()
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in tweet_words if word not in stop_words]
    filtered_tweet = " ".join(filtered_words)
    return filtered_tweet

# Function to remove HTML characters (e.g., &) from text
def remove_strayHTML(text):
    clean_text = text = re.sub(r'&\S+;', '', text)
    return clean_text

# Function to remove numbers from text
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

# Function to remove picture links (e.g., pic.twitter.com) from text
def remove_piclinks(text):
    return re.sub(r'pic\.twitter\.com/\S+', '', text)

# Function to remove short words (length <= 2) from text
def remove_shortwords(text):
    words = text.split()
    long_words = [word for word in words if len(word) > 2]
    return ' '.join(long_words)

## Testing Data Cleaning Functions (To be Removed)

In [13]:
def test_make_lowercase():
    assert make_lowercase("HeLLo!") == "hello!"
    assert make_lowercase("") == ""

def test_remove_mentions():
    assert remove_mentions("@bob hi") == " hi"
    assert remove_mentions("hi @bob") == "hi "
    assert remove_mentions("hi") == "hi"

def test_remove_punctuation():
    assert remove_punctuation("hi, world!") == "hi world"
    assert remove_punctuation("no_punct") == "nopunct"
    assert remove_punctuation("") == ""

def test_remove_url():
    assert remove_url("visit http://example.com now") == "visit  now"
    assert remove_url("visit https://example.com now") == "visit  now"
    assert remove_url("visit www.example.com now") == "visit  now"

def test_remove_extraspace():
    assert remove_extraspace("  hi   there  ") == "hi there"
    assert remove_extraspace("hi") == "hi"
    assert remove_extraspace("") == ""

def test_remove_stopwords():
    _ = stopwords.words("english")
    assert remove_stopwords("this is a test") == "test"
    assert remove_stopwords("the quick brown fox") == "quick brown fox"

def test_remove_strayHTML():
    assert remove_strayHTML("Tom &amp; Jerry") == "Tom  Jerry"
    assert remove_strayHTML("no html") == "no html"

def test_remove_numbers():
    assert remove_numbers("a1b2c3") == "abc"
    assert remove_numbers("2026-02-25") == "--"
    assert remove_numbers("no nums") == "no nums"

def test_remove_piclinks():
    assert remove_piclinks("check pic.twitter.com/abc") == "check "
    assert remove_piclinks("no pic link") == "no pic link"

def test_remove_shortwords():
    assert remove_shortwords("I am at NTU") == "NTU"
    assert remove_shortwords("to be or not to be") == "not"

def run_all_tests():
    tests = [
        test_make_lowercase,
        test_remove_mentions,
        test_remove_punctuation,
        test_remove_url,
        test_remove_extraspace,
        test_remove_stopwords,
        test_remove_strayHTML,
        test_remove_numbers,
        test_remove_piclinks,
        test_remove_shortwords,
    ]
    for t in tests:
        t()
    print(f"Passed {len(tests)} tests")

run_all_tests()

Passed 10 tests
